# Detecting & Mitigating Hallucinations — Hands-On Lab

**Tools covered live:** RAGAS (detect, offline) · SelfCheckGPT (detect, runtime, no context needed, open source) · Guardrails AI (mitigate, runtime)
**Covered conceptually (not run here):** NeMo Guardrails — see the last section.

**Learning objectives** — by the end of this notebook you'll be able to:
1. Explain the difference between *detecting* a hallucination after generation and *mitigating* it before a user sees it.
2. Run RAGAS's `faithfulness` metric on a small QA set and read the score.
3. Run SelfCheckGPT to get a self-consistency-based hallucination score with no retrieved context required.
4. Wire a Guardrails AI validator into an LLM call and watch it catch/fix a hallucination live.
5. Know where NeMo Guardrails fits in the picture even without running it here.

> **One-line takeaway:** RAGAS and SelfCheckGPT tell you a hallucination happened; Guardrails AI and NeMo Guardrails try to stop it from reaching the user. Production systems want at least one of each category.


## Setup

In [ ]:
%pip install ragas datasets langchain-google-genai google-genai guardrails-ai
%pip install selfcheckgpt spacy
!python -m spacy download en_core_web_sm


In [ ]:
import os
from getpass import getpass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("GEMINI_API_KEY (get one at https://aistudio.google.com/apikey): ")

from google import genai
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Swap this for whichever current Gemini model you have access to
# (gemini-2.0-flash is the cheap/fast default; gemini-2.5-flash or newer if you have it).
MODEL_NAME = "gemini-3.1-flash-lite"


In [ ]:
# Run this once BEFORE the session so the SelfCheckGPT NLI model (~500MB, from Hugging Face)
# is cached locally — you don't want that download happening live in front of a room.
from selfcheckgpt.modeling_selfcheck import SelfCheckNLI

selfcheck_nli = SelfCheckNLI(device="cpu")  # switch to "cuda" if you have a GPU on the demo machine


## The corpus

A **synthetic** insurance policy doc — not something the model already "knows" from pretraining, so retrieval actually matters and any invented fact is instantly checkable against the source text. Fictional company.


In [ ]:
policy_doc = """
AURORA HEALTH — OUTPATIENT (OPD) POLICY FAQ (v3)

1. OPD Sub-limit: ₹15,000 per policy year, shared across consultations,
   diagnostics, and pharmacy.
2. Waiting Period: OPD benefits activate after 30 days from policy start
   date, except accidental injury which is covered from day 1.
3. Claim Mode: Cashless only at network clinics listed in Annexure B.
   Reimbursement claims for OPD are not accepted.
4. Consultation Cap: Max ₹800 reimbursed per consultation, max 2
   consultations per specialty per month.
5. Diagnostics: Covered only if prescribed by a network doctor during a
   covered consultation. Self-initiated tests are excluded.
6. Pharmacy: Covered only for medicines prescribed against a covered
   consultation, capped at 7 days' dosage per prescription.
7. Exclusions: Dental (except accidental), vision correction, and
   cosmetic consultations are not covered under OPD.
8. Renewal: OPD sub-limit resets each policy year and does not carry
   forward unused balance.
"""

questions = [
    "What's the OPD sub-limit per year?",                                            # Q1 - in-scope, clean
    "Can I get reimbursed for an OPD consultation at a non-network clinic?",          # Q2 - in-scope, tempts a wrong "yes"
    "If I see a specialist 3 times in one month, how much gets reimbursed?",          # Q3 - multi-hop across clauses 4
    "What's the waiting period for maternity-related OPD visits?",                    # Q4 - OUT OF SCOPE, best hallucination bait
    "Since dental cleanings are covered under OPD, how do I file that claim?",        # Q5 - FALSE PREMISE, contradicts clause 7
]

print(f"{len(questions)} questions loaded.")

## Part 1 — Establish the baseline

First, ask the raw LLM with **no context at all**. Watch Q4 and Q5 — it will confidently invent a waiting period and a claims process that don't exist.


In [ ]:
def naive_answer(question, model=MODEL_NAME):
    response = client.models.generate_content(model=model, contents=question)
    return response.text

naive_answers = [naive_answer(q) for q in questions]

for q, a in zip(questions, naive_answers):
    print(f"Q: {q}\nA: {a}\n{'-'*80}")


Now re-run with the policy doc stuffed into the prompt as context — the simplest possible "RAG" (no retriever, no chunking, just the whole doc; that's fine for a demo this size).

Q1–Q3 should visibly improve. **Q4 and Q5 often still hallucinate**.


In [ ]:
def rag_answer(question, context, model=MODEL_NAME):
    prompt = f"""Answer the question using ONLY the context below. If the context doesn't cover it, say so explicitly.

Context:
{context}

Question: {question}"""
    response = client.models.generate_content(model=model, contents=prompt)
    return response.text

rag_answers = [rag_answer(q, policy_doc) for q in questions]

for q, a in zip(questions, rag_answers):
    print(f"Q: {q}\nA: {a}\n{'-'*80}")


## Part 2 — Detect with RAGAS

`faithfulness` breaks the answer into individual claims and checks each one against the retrieved context using an LLM judge — reference-free, no gold answers needed.

**Live callout:** Q1–Q3 should score high (~0.9+). Q4/Q5 should score low — faithfulness catches the invented maternity waiting period because it's a claim not entailed by the context. Frame this as *offline*: the thing you run on a golden set before shipping, or gate CI with — not something that runs per live request.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0,
))
evaluator_embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    google_api_key=os.environ["GEMINI_API_KEY"],
))


In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

ragas_data = {
    "question": questions,
    "answer": rag_answers,
    "contexts": [[policy_doc]] * len(questions),
}

ragas_dataset = Dataset.from_dict(ragas_data)
ragas_result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

ragas_df = ragas_result.to_pandas()
ragas_df[["question", "faithfulness", "answer_relevancy"]]


## Part 3 — Mitigate with Guardrails AI

Guardrails wraps the LLM call with a validator that checks groundedness and can reask, fix, or block on failure — **runtime**, not offline.

> Validator install paths change fairly often on the Guardrails Hub — if `guardrails_ai.provenance_embeddings` doesn't resolve, check [guardrailsai.com/hub](https://guardrailsai.com/hub) for the current package name for whichever hallucination/provenance validator you pick.



In [ ]:
# pip install for the validator (uncomment if not already installed):
# %pip install guardrails-ai sentence-transformers
# then grab the specific validator package per the Hub page, e.g.:
# %pip install guardrails-hub-provenance-embeddings

from guardrails import Guard
from guardrails_ai.provenance_embeddings import ProvenanceEmbeddings

guard = Guard().use(
    ProvenanceEmbeddings(
        validation_method="sentence",
        llm_callable=f"gemini/{MODEL_NAME}",   # LiteLLM's Gemini naming convention
        on_fail="fix",                          # swap to "exception" to see it hard-block instead
    )
)


In [ ]:
# Run it on the Q4 hallucination from the NAIVE (no-context) answer —
# the invented maternity waiting period should get caught/stripped in real time.
q4_naive = naive_answers[3]

result = guard.validate(
    q4_naive,
    metadata={"query": questions[3], "sources": [policy_doc]},
)

print("Original :", q4_naive)
print("\nPassed?  :", result.validation_passed)
print("Validated:", result.validated_output)


**Contrast:** RAGAS *told you* Q4 was wrong, after the fact. Guardrails AI *stops* the unsupported sentence from reaching the user, in the same call that produced it.
